In [10]:
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from dataloader import get_dataloaders
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import os

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, _, _, label_to_idx = get_dataloaders(data_dir="data", batch_size=32)
num_classes = len(label_to_idx)

print(f"class mapping: {label_to_idx}")
print(f"num_classes: {num_classes}")

class mapping: {'Amber złote lwy': 0, 'Dojlidy': 1, 'Heineken': 2, 'Other/none': 3, 'Perła Chmielowa': 4, 'Specjal': 5}
train batches: 12,  val batches: 3, test batches: 3


In [12]:
import random
from torchvision.utils import save_image

save_dir = "outputs/augmentation/example_transforms"
os.makedirs(save_dir, exist_ok=True)

ds = train_loader.dataset
for i, idx in enumerate(random.sample(range(len(ds)), 10)):
    img, _ = ds[idx]
    save_image(img, f"{save_dir}/{i}_{ds.data[idx]['image']}")

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_classes = len(label_to_idx)


class BeerResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        for param in self.backbone.parameters():
            param.requires_grad = False

        self.backbone.fc = nn.Sequential(
            nn.Linear(self.backbone.fc.in_features, 256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


model = BeerResNet(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.backbone.fc.parameters(), lr=1e-3)

Using device: cuda


In [14]:
def evaluate_model(model, train_loader, val_loader, criterion, device, num_classes):
    model.eval()
    results = {}

    with torch.no_grad():
        for split, loader in [("train", train_loader), ("val", val_loader)]:
            all_labels = []
            all_preds = []
            all_probs = []
            total_loss = 0.0

            for images, labels in loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                total_loss += loss.item() * images.size(0)
                probs = torch.softmax(outputs, dim=1)
                _, predicted = outputs.max(1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

            all_labels = np.array(all_labels)
            all_preds = np.array(all_preds)
            all_probs = np.array(all_probs)

            results[f"{split}_loss"] = total_loss / len(all_labels)
            results[f"{split}_accuracy"] = accuracy_score(all_labels, all_preds)
            results[f"{split}_f1_score"] = f1_score(all_labels, all_preds, average="weighted")
            try:
                results[f"{split}_roc_auc"] = roc_auc_score(
                    all_labels, all_probs, multi_class="ovr", average="weighted"
                )
            except ValueError:
                results[f"{split}_roc_auc"] = float("nan")
            results[f"{split}_confusion_matrix"] = confusion_matrix(
                all_labels, all_preds, labels=list(range(num_classes))
            )

    return results


def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_classes, num_epochs, trial=None):
    training_records = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_total += labels.size(0)
            pbar.set_postfix(train_loss=train_loss / train_total)
        pbar.close()

        results_record = evaluate_model(model, train_loader, val_loader, criterion, device, num_classes)
        results_record["epoch"] = epoch + 1
        training_records.append(results_record)

        print(
            f"\rEpoch {epoch+1}/{num_epochs} - "
            f"train_loss: {results_record['train_loss']:.4f} | "
            f"val_loss: {results_record['val_loss']:.4f}"
        )

        if trial is not None:
            trial.report(results_record["val_loss"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return training_records

In [15]:
def objective(trial):
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32, 64])
    num_filters_1 = trial.suggest_categorical("num_filters_1", [16, 32, 64])
    num_filters_2 = trial.suggest_categorical("num_filters_2", [32, 64, 128])
    hidden_size = trial.suggest_categorical("hidden_size", [128, 256, 512])
    dropout = trial.suggest_float("dropout", 0.2, 0.7)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    num_epochs = trial.suggest_int("num_epochs", 10, 30)

    train_loader, val_loader, _, _ = get_dataloaders(data_dir="data", batch_size=batch_size)

    model = BeerCNN(
        num_classes,
        num_filters_1=num_filters_1,
        num_filters_2=num_filters_2,
        hidden_size=hidden_size,
        dropout=dropout,
    ).to(device)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    training_records = train_model(
        model, train_loader, val_loader, criterion, optimizer,
        device, num_classes, num_epochs, trial=trial,
    )

    return training_records[-1]["val_loss"]


study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=20)

print(f"\nbest trial val_loss: {study.best_trial.value:.4f}")
print(f"best params: {study.best_trial.params}")

Epoch 1/20: 100%|██████████| 12/12 [00:01<00:00,  9.42it/s, train_loss=6.66]


val loss: 1.7768


Epoch 2/20: 100%|██████████| 12/12 [00:01<00:00, 10.07it/s, train_loss=1.79]


val loss: 1.7919


Epoch 3/20: 100%|██████████| 12/12 [00:01<00:00, 10.41it/s, train_loss=1.79]


...

val loss: 1.7841


Epoch 19/20: 100%|██████████| 12/12 [00:01<00:00, 10.33it/s, train_loss=1.75]


val loss: 1.7849


Epoch 20/20: 100%|██████████| 12/12 [00:01<00:00, 10.39it/s, train_loss=1.75]


val loss: 1.8005


In [17]:
best = study.best_trial.params

t_loader, v_loader, _, _ = get_dataloaders(data_dir="data", batch_size=best["batch_size"])

model = BeerCNN(
    num_classes,
    num_filters_1=best["num_filters_1"],
    num_filters_2=best["num_filters_2"],
    hidden_size=best["hidden_size"],
    dropout=best["dropout"],
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best["lr"])

training_records = train_model(
    model, t_loader, v_loader, criterion, optimizer,
    device, num_classes, best["num_epochs"],
)

run_name = input("Enter run name: ")
output_dir = os.path.join("outputs", run_name)
os.makedirs(output_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(output_dir, "model.pt"))

metrics_df = pd.DataFrame(training_records)
metrics_df.to_csv(os.path.join(output_dir, "metrics.csv"), index=False)

print(f"Saved to {output_dir}/")

Saved to outputs//
